# Week 4 Lab – Representing Text: from counts to contextual embeddings

In this lab you will compare the representations covered in the lecture, from sparse count vectors up to the kinds of representations used inside modern language models:

| Part | Representation | Key idea |
|---|---|---|
| 1 | Count vectors & TF-IDF | Sparse, one dimension per word type |
| 2 | GloVe word embeddings | Dense, static: one vector per word *type* |
| 3 | Subword tokenisation | How modern models split text *before* embedding it |
| 4 | Contextual embeddings (BERT) | One vector per word *occurrence* |
| 5 | Sentence embeddings | Models trained to embed whole sentences |
| 6 | Semantic search | Putting it all to work |

**Runtime:** everything runs on the free Colab CPU runtime (a GPU is not needed). Allow about 25 minutes.

Look for the **Explore** cells: these are the things to try. Where it says *predict first*, write your prediction down before you run the cell, because checking your intuition is what the lab is for.

## Setup
Run this cell first. It starts downloading GloVe in the background (it's ~800MB and takes 2–3 minutes), so it will be ready by the time you reach Part 2.

In [ ]:
import os, subprocess
if not os.path.exists('glove_ready'):
    subprocess.Popen(
        'wget -q -nc https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip && '
        'unzip -o -q glove.6B.zip glove.6B.100d.txt && rm -f glove.6B.zip && touch glove_ready',
        shell=True)
    print("GloVe download started in the background - carry on with Part 1.")
else:
    print("GloVe already downloaded.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
pd.set_option('display.precision', 2)
pd.set_option('display.width', 200)

def show_sim(matrix, labels, title):
    # Plot a cosine-similarity matrix as an annotated heat map
    plt.figure(figsize=(0.9 * len(labels) + 2, 0.7 * len(labels) + 1.5))
    sns.heatmap(matrix, xticklabels=labels, yticklabels=labels, annot=True, fmt='.2f',
                cmap='viridis', vmin=0, vmax=1, square=True, cbar=False)
    plt.title(title)
    plt.show()

## Part 1 – Sparse representations: counts and TF-IDF

This is the corpus used throughout the lab. It's the same as the lecture examples, plus **doc6**, which says roughly the same thing as doc1 but shares almost no words with it.

In [ ]:
doc1 = 'The cat is on the mat .'
doc2 = 'the wolf is on the mat'
doc3 = 'paris is a nice city to visit in the autumn'
doc4 = 'the man bit the dog'
doc5 = 'the dog bit the man'
doc6 = 'a kitten was sitting on the rug'
corpus = [doc1, doc2, doc3, doc4, doc5, doc6]
labels = [f'doc{i+1}' for i in range(len(corpus))]

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

count_vec = CountVectorizer()
X_count = count_vec.fit_transform(corpus)
print("The raw (sparse) matrix only stores the non-zero entries: (doc, word_index)  count")
print(X_count[:2])
print("\nAs a dense data frame (like a spreadsheet):")
df_count = pd.DataFrame(X_count.toarray(), columns=count_vec.get_feature_names_out(), index=labels)
display(df_count)
print(f"Sparsity: {1 - X_count.nnz / np.prod(X_count.shape):.0%} of the cells are zero")
show_sim(cosine_similarity(X_count), labels, 'Count vectors: cosine similarity')

In [ ]:
tfidf_vec = TfidfVectorizer()
X_tfidf = tfidf_vec.fit_transform(corpus)
df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=tfidf_vec.get_feature_names_out(), index=labels)
display(df_tfidf)
show_sim(cosine_similarity(X_tfidf), labels, 'TF-IDF: cosine similarity')

### Explore (Part 1)
1. doc4 and doc5 have a similarity of exactly 1.00 under both methods. Why? What does that tell you about bag-of-words?
2. What is the similarity between doc1 and doc6? Is that what a human would say?
3. Compare the weight of *the* in the count and TF-IDF tables. Why has TF-IDF shrunk it?

## Part 2 – Static word embeddings (GloVe)

Each word *type* gets one dense 100-dimensional vector, learned from co-occurrence statistics over Wikipedia and newswire text. We load the 100,000 most frequent words.

In [ ]:
import time
while not os.path.exists('glove_ready'):
    print("Waiting for GloVe download to finish..."); time.sleep(10)

glove = {}
with open('glove.6B.100d.txt', encoding='utf8') as f:
    for i, line in enumerate(f):
        if i >= 100_000:
            break
        parts = line.rstrip().split(' ')
        glove[parts[0]] = np.asarray(parts[1:], dtype='float32')
print(f"Loaded {len(glove):,} word vectors of dimension {len(glove['cat'])}")
print("First 10 numbers of 'cat':", glove['cat'][:10])

# Normalised matrix for fast nearest-neighbour search
glove_words = list(glove.keys())
G = np.stack([glove[w] for w in glove_words])
G = G / np.linalg.norm(G, axis=1, keepdims=True)

def glove_sim(w1, w2):
    return float(cosine_similarity([glove[w1]], [glove[w2]])[0, 0])

def nearest(vec_or_word, n=8, exclude=()):
    if isinstance(vec_or_word, str):
        v, exclude = glove[vec_or_word], set(exclude) | {vec_or_word}
    else:
        v = vec_or_word
    sims = G @ (v / np.linalg.norm(v))
    out = []
    for i in np.argsort(-sims):
        if glove_words[i] not in exclude:
            out.append((glove_words[i], round(float(sims[i]), 2)))
        if len(out) == n:
            break
    return out

def analogy(a, b, c, n=5):
    # a is to b as c is to ?   e.g. man : king :: woman : ?
    return nearest(glove[b] - glove[a] + glove[c], n, exclude={a, b, c})

print("\nNearest to 'cat':", nearest('cat'))
print("man : king :: woman : ?", analogy('man', 'king', 'woman'))

In [ ]:
# Word-to-word similarity as a heat map
words = ['cat', 'kitten', 'dog', 'wolf', 'mat', 'rug', 'paris', 'dublin', 'autumn']
W = np.stack([glove[w] for w in words])
show_sim(cosine_similarity(W), words, 'GloVe word similarity')

# The same vectors squashed into 2D with PCA
from sklearn.decomposition import PCA
xy = PCA(n_components=2).fit_transform(W)
plt.figure(figsize=(6, 4))
plt.scatter(xy[:, 0], xy[:, 1])
for w, (x, y) in zip(words, xy):
    plt.annotate(w, (x, y), xytext=(4, 4), textcoords='offset points')
plt.title('GloVe vectors projected to 2D (PCA)')
plt.show()

Now build a document vector by **averaging** the word vectors (lecture slide: *From Words to Sentences*).

In [ ]:
def glove_doc(text):
    tokens = [t for t in text.lower().split() if t in glove]
    return np.mean([glove[t] for t in tokens], axis=0)

E_glove = np.stack([glove_doc(d) for d in corpus])
show_sim(cosine_similarity(E_glove), labels, 'Averaged GloVe: cosine similarity')

### Explore (Part 2)
1. Try `nearest(...)` and `analogy(...)` on your own words, e.g. `analogy('france', 'paris', 'ireland')`, `analogy('good', 'better', 'bad')`. Find one analogy that works and one that fails.
2. **Bias check:** try `analogy('man', 'doctor', 'woman')` and `analogy('man', 'programmer', 'woman')`. Where do you think this comes from?
3. doc1 vs doc6: did averaged GloVe do better than TF-IDF? Did it fix doc4 vs doc5? Why or why not?
4. Try `glove_sim('hot', 'cold')`. Antonyms often come out *very* similar. Why would co-occurrence-based training produce that?

## Part 3 – Subword tokenisation: what modern models actually see

GloVe has a fixed vocabulary, so any word not in it is simply lost (*out-of-vocabulary*). Modern models avoid this by splitting text into **subword tokens** learned from data (WordPiece for BERT, byte-level BPE for GPT-style models). Any string can then be represented, including typos, names and emoji.

In [ ]:
from transformers import AutoTokenizer
tokenizers = {
    'BERT (WordPiece, 30k)':  AutoTokenizer.from_pretrained('bert-base-uncased'),
    'GPT-2 (BPE, 50k)':        AutoTokenizer.from_pretrained('gpt2'),
    'Qwen2.5 (BPE, 151k)':     AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B'),
}

def show_tokens(text):
    print(f'\n"{text}"')
    for name, tk in tokenizers.items():
        ids = tk(text, add_special_tokens=False)['input_ids']
        pieces = [tk.decode([i]) for i in ids]
        print(f"  {name:24s} {len(ids):3d} tokens  {pieces}")

for text in ['The cat is on the mat.',
             'unbelievably',
             'Grangegorman',
             'The cat sat on teh mat lol 😂',
             'Tá an cat ar an mata.']:
    show_tokens(text)

print("\nIn GloVe?", {w: (w in glove) for w in ['grangegorman', 'teh', 'lol', 'tá']})

### ERxplore (Part 3)
1. Try `show_tokens(...)` with your name, a long compound word, a URL, some code, and a sentence in another language you know. Which tokeniser is most *efficient* (fewest tokens) and for what kind of text?
2. The Irish sentence usually costs far more tokens than its English equivalent. Since LLM APIs charge (and limit context) per token, what does this mean for speakers of less-resourced languages?
3. (Link to the lecture on encodings.) Byte-level BPE can represent *any* UTF-8 string. Why does working at the byte level guarantee there are no unknown tokens?

## Part 4 – Contextual embeddings: one vector per *occurrence*

In GloVe, *bank* has one vector whether we mean a river bank or a bank account. A transformer model such as BERT produces a different vector for each occurrence, shaped by the surrounding words.

In [ ]:
import torch
from transformers import AutoModel
bert_tok = tokenizers['BERT (WordPiece, 30k)']
bert = AutoModel.from_pretrained('bert-base-uncased').eval()

def word_in_context(sentence, word):
    # BERT vector for `word` in `sentence` (subword pieces are averaged)
    enc = bert_tok(sentence, return_tensors='pt')
    with torch.no_grad():
        hidden = bert(**enc).last_hidden_state[0]
    toks = bert_tok.convert_ids_to_tokens(enc['input_ids'][0])
    pieces = bert_tok.tokenize(word)
    for i in range(len(toks) - len(pieces) + 1):
        if toks[i:i + len(pieces)] == pieces:
            return hidden[i:i + len(pieces)].mean(0).numpy()
    raise ValueError(f"'{word}' not found in: {toks}")

def compare_senses(word, sentences):
    V = np.stack([word_in_context(s, word) for s in sentences])
    short = [s if len(s) < 40 else s[:37] + '...' for s in sentences]
    show_sim(cosine_similarity(V), short, f"BERT vectors for '{word}' in context")

compare_senses('bank', [
    'We had a picnic on the river bank.',
    'The boat drifted towards the bank of the river.',
    'I need to lodge money in the bank.',
    'The bank refused to give me a loan.',
])

### Explore (Part 4)
1. With GloVe, what would every cell in that heat map be? (Check: `glove_sim('bank', 'bank')`.)
2. Pick another ambiguous word (*bat*, *pitch*, *match*, *spring*, *mouse*, *cell*) and write 2 sentences per sense. Does BERT separate the senses? Find a case where it struggles.
3. Try the same word with the same sense in a formal sentence and a casual, dialogue-style utterance (e.g. *"I'm off to the bank"* vs *"The bank is located on Main Street"*). How much does register shift the vector?

## Part 5 – Sentence embeddings

Averaging word vectors (Part 2) is crude. Modern **sentence-embedding** models are transformers *fine-tuned* so that sentences with similar meanings end up close together. We'll use `all-MiniLM-L6-v2` (small and fast, 384 dimensions) and compare it with simply averaging BERT's outputs.

**Predict first:** before running the next two cells, copy this table into a text cell and fill in your guess (high / medium / low) for each pair and method.

| Pair | Count | TF-IDF | GloVe avg | BERT avg | MiniLM |
|---|---|---|---|---|---|
| doc1 vs doc2 (cat/wolf on mat) | | | | | |
| doc4 vs doc5 (man bit dog / dog bit man) | | | | | |
| doc1 vs doc6 (cat on mat / kitten on rug) | | | | | |
| doc1 vs doc3 (cat on mat / Paris) | | | | | |

In [ ]:
from sentence_transformers import SentenceTransformer
minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def bert_avg(sentences):
    enc = bert_tok(sentences, padding=True, return_tensors='pt')
    with torch.no_grad():
        h = bert(**enc).last_hidden_state
    mask = enc['attention_mask'].unsqueeze(-1)
    return ((h * mask).sum(1) / mask.sum(1)).numpy()

E_bert = bert_avg(corpus)
E_minilm = minilm.encode(corpus)
print("MiniLM embedding shape:", E_minilm.shape)
show_sim(cosine_similarity(E_minilm), labels, 'MiniLM sentence embeddings: cosine similarity')

In [ ]:
methods = {
    'Count': cosine_similarity(X_count),
    'TF-IDF': cosine_similarity(X_tfidf),
    'GloVe avg': cosine_similarity(E_glove),
    'BERT avg': cosine_similarity(E_bert),
    'MiniLM': cosine_similarity(E_minilm),
}
pairs = {'doc1 vs doc2': (0, 1), 'doc4 vs doc5': (3, 4), 'doc1 vs doc6': (0, 5), 'doc1 vs doc3': (0, 2)}
summary = pd.DataFrame({m: {p: S[i, j] for p, (i, j) in pairs.items()} for m, S in methods.items()})
display(summary)

### Explore (Part 5)
1. How did your predictions compare? Which method best matches human judgement on each pair?
2. Look at the *BERT avg* column: every pair is fairly similar, even the unrelated ones. Why might a model that was never trained to compare sentences give everything a high cosine score? (This is why sentence-level training matters: lecture slide *Sentence Embeddings*.)
3. Even MiniLM rates *man bit dog* and *dog bit man* as very similar. Is it still a bag of words? Test it: `cosine_similarity(minilm.encode(['the man bit the dog', 'the man was bitten by the dog', 'the dog bit the man']))`.
4. **Negation:** compare *"The film was good"*, *"The film was not good"* and *"The film was great"*. What does this tell you about using embedding similarity for sentiment?
5. **Dialogue:** embed some conversational turns, e.g. *"yeah"*, *"uh-huh"*, *"sure, sounds good"*, *"no way"*, *"I agree completely"*. Does the model capture what these utterances *do* in a conversation, or just their surface form?

## Part 6 – Putting it to work: semantic search

A very common use of sentence embeddings is retrieval: embed a set of documents once, then find the ones closest to a query. This is the "R" in *retrieval-augmented generation* (RAG) systems built around LLMs. Here we compare keyword search (TF-IDF) with embedding search.

In [ ]:
docs = [
    'The library opens at 8am and closes at 10pm during term.',
    'Students can borrow up to ten books at a time.',
    'Exam results are published on the student portal in June.',
    'The canteen serves hot meals from noon until 2pm.',
    'Parking permits must be renewed at the start of each academic year.',
    'Late assignments lose 10% of their marks per day.',
    'The gym is free for all registered students.',
    'Contact the IT help desk if you cannot log in to Moodle.',
]

def search(query, k=3):
    tv = TfidfVectorizer().fit(docs + [query])
    tf_scores = cosine_similarity(tv.transform([query]), tv.transform(docs))[0]
    em_scores = cosine_similarity(minilm.encode([query]), minilm.encode(docs))[0]
    print(f'\nQUERY: "{query}"')
    for name, s in [('TF-IDF', tf_scores), ('MiniLM', em_scores)]:
        print(f'  {name}:')
        for i in np.argsort(-s)[:k]:
            print(f'    {s[i]:.2f}  {docs[i]}')

search('When can I get something to eat?')
search('What happens if I hand in my project a few days late?')
search('my password is not working')

### Explore (Part 6)
1. Write three queries of your own: one that shares words with a document, one that is a pure paraphrase, and one that is off-topic (e.g. *"what's the weather like?"*). When does TF-IDF win? Does embedding search ever return a confident but wrong answer?
2. Embedding search always returns *something*. How might you decide when the best match isn't good enough to show the user?
3. **(Stretch)** Swap in the multilingual model `SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')` and query in French or Irish against the English documents. How well does cross-lingual search work for each language?

## Wrap-up
In two or three sentences, answer: **for a task of your choice (spam filtering, FAQ chatbot, plagiarism detection, dialogue act classification...), which representation from today would you pick, and why?** Think about accuracy, speed, interpretability and the kind of text involved.